# $$\text{Modelado del Churn}$$

# REGRESIÓN LOGÍSTICA

## 0. Configuración inicial

In [ ]:
# importar librerías

# Manipulación de datos
import pandas as pd
import numpy as np

# Manipulación de gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# Paths y automatizaciones
import joblib
import sys
from pathlib import  Path

# Módulos sklearn

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve,
    make_scorer, recall_score, # para scoring personalizado en GridSearchCV
    precision_score, f1_score 
)

# Explicabilidad modelo
import shap

In [ ]:
# Paths

INPUT_PATH   = Path("data\processed\df_final.csv")
MODEL_PATH   = Path("models\pipeline.pkl")
# METRICS_PATH = Path("4_outputs/synthetic_data/2_metrics")
FIG_PATH     = Path("figures")

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
# METRICS_PATH.mkdir(parents=True, exist_ok=True)
FIG_PATH.mkdir(parents=True, exist_ok=True)

SEED = 42

In [ ]:
# Importar dataframe para modelado
df_final = pd.read_csv("../data/processed/df_final.csv")

## 1. Carga de dataframe de usuarios

In [ ]:
# ── 1. Carga ──────────────────────────────────────────────────────────────────

df = pd.read_csv(INPUT_PATH, low_memory=False)
print("=" * 60)
print("REPORTE DE MODELADO — CHURN PREDICTION")
print("=" * 60)
print(f"\nFilas cargadas  : {len(df):,}")
print(f"Tasa de churn   : {df['churned'].mean():.2%}")

In [ ]:
# ── 2. Winsorización de variables con datos atípicos (percentil 99) ────────────────────────

# p99 = df["valor_total_6m"].quantile(0.99)
# df["valor_total_6m"] = df["valor_total_6m"].clip(upper=p99)
# print(f"\n── Winsorización valor_total_6m ───────────────────────────")
# print(f"  Percentil 99   : {p99:,.0f}")
# print(f"  Valor máximo post-winsorización: {df['valor_total_6m'].max():,.0f}")

## 2. Features del modelo

In [ ]:
# ── 1. Definir X e y ──────────────────────────────────────────────────────────
EXCLUIR = ["UserID", "churned"]
X = df.drop(columns=EXCLUIR)
y = df["churned"]

> 👇👇 ***AJUSTAR***

In [ ]:
# -- 2. Columnas por tipo -------------------------------------------------------------
NUMERICAS = [
    "antiguedad_dias", "num_envios_6m", "valor_total_6m",
    "ticket_promedio", "dias_ultimo_envio", "num_reclamos_6m",
    "tasa_entrega_exitosa", "nps_score", "ratio_reclamos_envios"
]
CATEGORICAS = [
    "ciudad", "tipo_cliente", "canal_principal",
    "tiene_contrato", "segmento_recencia"
]

print(f"\n── Features del modelo ────────────────────────────────────")
print(f"  Numéricas   ({len(NUMERICAS)}): {NUMERICAS}")
print(f"  Categóricas ({len(CATEGORICAS)}): {CATEGORICAS}")

In [ ]:
# ── 3. Split train/test estratificado (70% train set / 30% test set) ─────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X[NUMERICAS + CATEGORICAS], y,
    test_size=0.30, # 30% para test, 70% para train
    random_state=SEED,
    stratify=y # Mantener proporción de churn en ambos sets
)
print(f"\n── Split train/test ───────────────────────────────────────")
print(f"  Train : {len(X_train):,} filas | churn={y_train.mean():.2%}")
print(f"  Test  : {len(X_test):,}  filas | churn={y_test.mean():.2%}")

> **Column transformer**

- Se aplicó previamente el `OneHotEncoder` a las columnas categóricas.
- Sólo se aplicará a través del *Column Transformer*, el `StandardScaler` a las columnas con valores numéricos contínuos

In [ ]:
# ── 4. ColumnTransformer ──────────────────────────────────────────────────────
preprocesamiento = ColumnTransformer(transformers=[
    ("num", StandardScaler(), NUMERICAS)
])

## 3. Entrenamiento del modelo

- Se usa GridSearchCV para encontrar los mejores hiperparámetros de la regresión logística
- priorizamos `recall score` para capturar la mayor cantidad de churners posibles
- Se utiliza `cross-validation`, para robustecer el proceso de aprendizaje

In [ ]:
# ── 1. Pipeline + GridSearchCV ────────────────────────────────────────────────
# Se usa GridSearchCV para encontrar los mejores hiperparámetros de la regresión 
# logística
pipeline = Pipeline(steps=[
    ("preprocesamiento", preprocesamiento),
    ("modelo", GridSearchCV(
        estimator=LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        max_iter=1000,
        random_state=SEED
    ),
        param_grid={
            "C"        : [0.01, 0.1, 1, 10, 100],
            "l1_ratio" : [0.1, 0.5, 0.9] # 0.0=L2, 1.0=L1, 0.5=ElasticNet
        },
        scoring=make_scorer(recall_score, pos_label=1),
 # priorizamos recall para capturar la mayor cantidad de churners posibles
        cv=5, # 5-fold cross-validation, robustece el aprendizaje
        n_jobs=-1,
        verbose=1,
    ))
])

print(f"\n── Entrenando pipeline (GridSearchCV, 5-fold cross-validation, scoring=recall)...")
pipeline.fit(X_train, y_train)

In [ ]:
# ── 2. Extraer los Mejores parámetros ─────────────────────────────────────────────────────
best_params = pipeline.named_steps["modelo"].best_params_
best_score  = pipeline.named_steps["modelo"].best_score_
print(f"\n── Mejores parámetros ─────────────────────────────────────")
for k, v in best_params.items():
    print(f"  {k:12s}: {v}")
print(f"  Recall CV (train): {best_score:.4f}")

## 4. Evaluación del modelo

In [ ]:
# ── 1. Evaluación del modelo en test set ─────────────────────────────────────────────────
y_pred      = pipeline.predict(X_test)
y_prob      = pipeline.predict_proba(X_test)[:, 1]
auc         = roc_auc_score(y_test, y_prob)

print(f"\n── Métricas en test ───────────────────────────────────────")
print(f"  AUC-ROC : {auc:.4f}")
print(f"\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Activo","Churned"]))

In [ ]:
# ── 2. Matriz de confusión ─────────────────────────────────────────────────

cm       = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

## 5. Métricas de negocio

> 👇👇 **Ajustar**

> **Métricas de impacto para el negocio - Supuestos**

In [ ]:
# ── 2. Métricas de impacto para el negocio ───────────────────────────────────
print(f"\n── Métricas de impacto de negocio ─────────────────────────")
# NOTA: Los siguientes valores son supuestos estimados.
# En producción deben ser provistos por el área financiera/comercial
# de Interrapidísimo.
SUPUESTOS = {
# se puede calcular como el valor total de compras en 6 meses multiplicado por un factor de retención anual
# (ej. 2x para estimar 1 año)
"LTV_PROMEDIO"      : df["valor_total_6m"].median() * 2,  # estimado anual
# COP por cliente contactado por multiples canales (teléfono, email, SMS)
# Tiempo agente call center (15 min) = 7,000 COP
# Descuento o incentivo ofrecido = 5,000 COP
# Costo operativo (SMS, email, sistema) = 3,000 COP
"COSTO_INTERVENCION" : 15_000,
"TASA_RETENCION"      : 0.30,     # % de clientes contactados que se retienen
}

print(f"\n── Supuestos de negocio (configurables) ───────────────────")
for k, v in SUPUESTOS.items():
    print(f"  {k:25s}: {v:,.2f}")

In [ ]:
# Métricas de negocio
clientes_contactados = tp + fp
clientes_retenidos   = int(clientes_contactados * SUPUESTOS["TASA_RETENCION"])
costo_total          = clientes_contactados * SUPUESTOS["COSTO_INTERVENCION"]
ingreso_retenido     = clientes_retenidos   * SUPUESTOS["LTV_PROMEDIO"]
roi                  = (ingreso_retenido - costo_total) / costo_total * 100

print(f"\n── Métricas de impacto ────────────────────────────────────")
print(f"  Clientes churn detectados (TP) : {tp:,}")
print(f"  Falsos positivos (FP)          : {fp:,}")
print(f"  Clientes a contactar           : {clientes_contactados:,}")
print(f"  Clientes retenidos estimados   : {clientes_retenidos:,}")
print(f"  Costo total campaña            : ${costo_total:,.0f} COP")
print(f"  Ingreso retenido estimado      : ${ingreso_retenido:,.0f} COP")
print(f"  ROI estimado del modelo        : {roi:.1f}%")
print(f"\n  ⚠ Supuestos estimados — validar con áreas comercial y financiera para mayor precisión en métricas de negocio")


## 6. Gráficas

In [ ]:
# ── 11. Gráficas ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Evaluación del Modelo — Churn Prediction", fontsize=13, fontweight="bold")

# 11.1 Matriz de confusión
sns.heatmap(
    confusion_matrix(y_test, y_pred),
    annot=True, fmt="d", cmap="Blues",
    xticklabels=["Activo","Churn"],
    yticklabels=["Activo","Churn"],
    ax=axes[0]
)
axes[0].set_title("Matriz de Confusión")
axes[0].set_ylabel("Real")
axes[0].set_xlabel("Predicho")

# 11.2 Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color="#e74c3c", lw=2, label=f"AUC = {auc:.3f}")
axes[1].plot([0,1],[0,1], "k--", lw=1)
axes[1].set_title("Curva ROC")
axes[1].set_xlabel("Tasa Falsos Positivos")
axes[1].set_ylabel("Tasa Verdaderos Positivos")
axes[1].legend()

# 11.3 Curva Precision-Recall
precision, recall, _ = precision_recall_curve(y_test, y_prob)
axes[2].plot(recall, precision, color="#3498db", lw=2)
axes[2].set_title("Curva Precision-Recall")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")

plt.tight_layout()
fig.savefig(FIG_PATH / "02_model_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\nGráfica guardada en: {FIG_PATH / '02_model_evaluation.png'}")

## 7. Pipeline

In [ ]:
# ── 12. Guardar pipeline ──────────────────────────────────────────────────────
joblib.dump(pipeline, MODEL_PATH)
print(f"\n── Pipeline guardado en: {MODEL_PATH}")

In [ ]:
# ── 13. Verificar pipeline desde .pkl ────────────────────────────────────────
modelo_cargado = joblib.load(MODEL_PATH)
muestra_pred   = modelo_cargado.predict(X_test.iloc[:5])
print(f"  Verificación predict() desde .pkl: {muestra_pred}")

# EXPLICABILIDAD LOGIT - SHAP VALUES

Explica las predicciones del modelo de regresión logística usando valores SHAP (SHapley Additive exPlanations).

In [ ]:
# ── 3. Cargar pipeline ────────────────────────────────────────────────────────
pipeline = joblib.load(MODEL_PATH)
print(f"\nPipeline cargado desde: {MODEL_PATH}")

# ── 4. Transformar X con el preprocesador del pipeline ────────────────────────
preprocesador = pipeline.named_steps["preprocesamiento"]
X_transformado = preprocesador.transform(X)

# Recuperar nombres de columnas tras One Hot Encoding
ohe_features = preprocesador.named_transformers_["cat"]\
    .get_feature_names_out(CATEGORICAS).tolist()
feature_names = ohe_features + NUMERICAS

print(f"\nFeatures tras preprocesamiento : {len(feature_names)}")
print(f"  Categóricas (OHE) : {len(ohe_features)}")
print(f"  Numéricas         : {len(NUMERICAS)}")


# ── 5. Extraer modelo base del GridSearchCV ───────────────────────────────────
mejor_modelo = pipeline.named_steps["modelo"].best_estimator_

In [ ]:
# ── 6. Calcular SHAP values ───────────────────────────────────────────────────
# Muestra de 50 filas para eficiencia
np.random.seed(SEED)
idx_muestra    = np.random.choice(len(X_transformado), size=50, replace=False)
X_muestra      = X_transformado[idx_muestra]

explainer   = shap.LinearExplainer(mejor_modelo, X_muestra)
shap_values = explainer.shap_values(X_muestra)

print(f"\nSHAP values calculados sobre muestra de 50 filas")


In [ ]:
# ── 7. Importancia global (mean |SHAP|) ───────────────────────────────────────
importancia = pd.DataFrame({
    "feature"   : feature_names,
    "mean_shap" : np.abs(shap_values).mean(axis=0)
}).sort_values("mean_shap", ascending=False).reset_index(drop=True)

print(f"\n── Top 15 variables por importancia SHAP ──────────────────")
print(importancia.head(15).to_string())

Gráficos SHAP Values

In [ ]:
# ── 8. Gráficas ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Análisis SHAP — Churn Prediction", fontsize=13, fontweight="bold")

# 8.1 Bar plot importancia global
top15 = importancia.head(15)
axes[0].barh(top15["feature"][::-1], top15["mean_shap"][::-1], color="#3498db")
axes[0].set_title("Importancia Global (mean |SHAP|)")
axes[0].set_xlabel("mean(|SHAP value|)")

# 8.2 Beeswarm manual (dot plot por feature)
top10_features = importancia.head(10)["feature"].tolist()
top10_idx      = [feature_names.index(f) for f in top10_features]
shap_top10     = shap_values[:, top10_idx]

for i, (feat, shap_col) in enumerate(zip(top10_features, shap_top10.T)):
    y_jitter = np.random.normal(i, 0.08, size=len(shap_col))
    axes[1].scatter(shap_col, y_jitter, alpha=0.3, s=8, c="#e74c3c")

axes[1].set_yticks(range(len(top10_features)))
axes[1].set_yticklabels(top10_features)
axes[1].axvline(0, color="black", lw=0.8, linestyle="--")
axes[1].set_title("Distribución SHAP — Top 10 Features")
axes[1].set_xlabel("SHAP value (impacto en predicción de churn)")

plt.tight_layout()
fig.savefig(FIG_PATH / "03_shap_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\nGráfica guardada en: {FIG_PATH / '03_shap_analysis.png'}")

Resultados

In [ ]:
# ── 9. Interpretación resultados ───────────────────────────────────────────────
top3 = importancia.head(3)["feature"].tolist()
print(f"\n── Interpretación de resultados ───────────────────────────")
print(f"  Las 3 variables más influyentes en la predicción de churn:")
for i, feat in enumerate(top3, 1):
    shap_mean = importancia.loc[importancia["feature"]==feat, "mean_shap"].values[0]
    print(f"  {i}. {feat:35s} mean|SHAP|={shap_mean:.4f}")
print(f"""
  Interpretación general:
  - Valores SHAP positivos → aumentan la probabilidad de churn
  - Valores SHAP negativos → disminuyen la probabilidad de churn
  - mean|SHAP| alto → variable muy influyente en la explicabilidad del modelo
""")

# 🅿SEGMENTACIÓN USUARIOS (CLUSTERING)